In [1]:
import pandas as pd
import numpy as np
from utils import create_samples , models_comparison_metrics , train_test_models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from itertools import combinations_with_replacement

print("Libraries loaded")

Libraries loaded


In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv("loan.csv" , 
                 usecols = ['loan_status' , 'annual_inc' , 'loan_amnt' , 'int_rate' , 'installment' , 'issue_d'] ,
                 nrows = 800_000
                )
df.shape

(800000, 6)

In [4]:
df = df[df['loan_status'].isin(['Fully Paid' , 'Charged Off'])]

In [5]:
df['y'] = (df['loan_status'] == 'Fully Paid').astype(int)

In [6]:
df.shape

(244353, 7)

In [7]:
df = df.dropna()

In [8]:
test_size = 100000

df_train = df.iloc[test_size : ]
df_test = df.iloc[ : test_size]

In [9]:
features = ['annual_inc' , 'loan_amnt' , 'int_rate' , 'installment']
target = 'y'

X_test = df_test[features]
y_test = df_test[target]

In [10]:
SEEDS = range(5)

In [11]:
models = [
    ("logistic" , LogisticRegression(max_iter = 2000)) ,
    ("tree" , DecisionTreeClassifier(max_depth = 5)) ,
    ("forest" , RandomForestClassifier(n_estimators = 100 , max_depth = 5))
]

train_sizes = [500 , 1000 , 3000 , 10000 , 70000]
data_overlaps = [0 , 0.25 , 0.5 , 0.75 , 1]

In [12]:
results = []
counter = 0

for (m1_name , m1) , (m2_name , m2) in combinations_with_replacement(models , 2):
    for sample_size in train_sizes:
        for data_overlap in data_overlaps:
            for exp_number, seed in enumerate(SEEDS, start=1):
                counter += 1
                if counter % 25 == 0:
                    print(f"Completed {counter}/750")
                
                model1 = clone(m1).set_params(random_state = seed)
                model2 = clone(m2).set_params(random_state = seed)

                X_train_1 , y_train_1 , X_train_2 , y_train_2 , obv_MAD , obv_MAD_to_average = create_samples(
                    df_train , sample_size , data_overlap , seed)

                y_pred_1 , y_prob_1 , y_pred_2 , y_prob_2 = train_test_models(
                    model1 , model2 , X_train_1 , y_train_1 , X_train_2 , y_train_2 , X_test)

                exp_results = models_comparison_metrics(
                    m1_name , m2_name , y_pred_1 , y_prob_1 , y_pred_2 , y_prob_2 , y_test , sample_size , data_overlap , exp_number , obv_MAD , obv_MAD_to_average , seed)

                
                results.append(exp_results)




results_df = pd.DataFrame(results)
print("Experiment finished, results are loaded")

Completed 25/750
Completed 50/750
Completed 75/750
Completed 100/750
Completed 125/750
Completed 150/750
Completed 175/750
Completed 200/750
Completed 225/750
Completed 250/750
Completed 275/750
Completed 300/750
Completed 325/750
Completed 350/750
Completed 375/750
Completed 400/750
Completed 425/750
Completed 450/750
Completed 475/750
Completed 500/750
Completed 525/750
Completed 550/750
Completed 575/750
Completed 600/750
Completed 625/750
Completed 650/750
Completed 675/750
Completed 700/750
Completed 725/750
Completed 750/750
Experiment finished, results are loaded


In [13]:
results_df.to_csv("results.csv", index=False)

In [14]:
group_cols = ["model1", "model2", "sample_size", "data_overlap"]

metric_cols = [
    "pred_corr",
    "prob_corr",
    "P_1_1",
    "P_1_0",
    "P_0_1",
    "P_0_0",
    "agreement_rate",
    "jaccard",
    "err_corr",
    "m1_fn_rate",
    "m1_fp_rate",
    "m2_fn_rate",
    "m2_fp_rate",
    "expected_FN_both",
    "empirical_FN_both",
    "FN_lift",
    "FN_excess",
    "FN_corr",
    "FN_jaccard",
    "MAD",
    "MAD_to_average"
]

results_aggregate = (
    results_df
    .groupby(group_cols)[metric_cols].mean()
    .reset_index()
)

In [15]:
results_aggregate.to_csv("results_aggregate.csv", index=False)